## Example LLM Chatbot with Ollama and Weaviate

This notebook does the following:
- Pulls Ollama models (embed + chat)
- Connects to a (local or remote) Weaviate v4 server
- Creates a RAG collection (BYO vectors)
- Embeds docs with Ollama and stores them in Weaviate
- Lists stored objects
- Queries via vector search
- Uses the results for RAG with the chat model
- Deletes the data in Weaviate
- Deletes the Ollama models and lists remaining ones

You can access the [Weaviate Documentation](http://docs.weaviate.io/weaviate) and the [Ollama Documentation](https://docs.ollama.com) for more information.
- The [Weaviate Client for Python](https://docs.weaviate.io/weaviate/client-libraries/python) is used in this example.

This notebook also shows you how to query the SQL Server instance from Python.
- It uses SQL Alchemy to fetch some data from the customer table in the Viskan database.
- You can access the [SQL Alchemy Documentation](https://docs.sqlalchemy.org/en/20) for more information.

## Imports

In [23]:
import os
import time
import json
import textwrap
from typing import List, Dict, Any

import requests
import weaviate
from weaviate.classes.config import Property, DataType, Configure
from weaviate.classes.init import Auth

from sqlalchemy import create_engine, text

## Configuration (Ollama and Weaviate)
- Change the `COLLECTION_NAME` in the cell below (at least change the group number to your group number), before executing the cell.

In [ ]:
# ==== OLLAMA CONFIG ====
OLLAMA_HOST = "http://127.0.0.1:11434"
EMBEDDING_MODEL = "nomic-embed-text"
CHAT_MODEL = "llama3:8b"             # or "llama3" or "phi3:mini", etc.

# ==== WEAVIATE CONFIG ====
WEAVIATE_HTTP_HOST = "172.31.42.91" # IP or DNS name
WEAVIATE_HTTP_PORT = 8080           # HTTP port
WEAVIATE_HTTP_SECURE = False        # True if https, False if http
WEAVIATE_GRPC_HOST = "172.31.42.91" # IP or DNS name
WEAVIATE_GRPC_PORT = 50051          # gRPC port
WEAVIATE_GRPC_SECURE = False        # True if TLS on gRPC, else False
WEAVIATE_API_KEY = None             # or "your-api-key"

# Collection name for the RAG example
# - Weaviate collection names have strict rules:
#   - Must start with an uppercase letter
#   - Can only contain: letters, digits, and underscores
#   - No hyphens/dashes, spaces, or other symbols
COLLECTION_NAME = "Group1RagDocument"

## Connect to Weaviate

In [5]:
# ==== WEAVIATE CLIENT SETUP ====

connect_kwargs = {
    "http_host": WEAVIATE_HTTP_HOST,
    "http_port": WEAVIATE_HTTP_PORT,
    "http_secure": WEAVIATE_HTTP_SECURE,
    "grpc_host": WEAVIATE_GRPC_HOST,
    "grpc_port": WEAVIATE_GRPC_PORT,
    "grpc_secure": WEAVIATE_GRPC_SECURE,
}

if WEAVIATE_API_KEY:
    connect_kwargs["auth_credentials"] = Auth.api_key(WEAVIATE_API_KEY)

client = weaviate.connect_to_custom(**connect_kwargs)

print("Weaviate ready?", client.is_ready())

Weaviate ready? True


## Ollama Helper Functions

In [ ]:
def ollama_pull_model(model: str):
    """Pull an Ollama model using the CLI."""
    print(f"Pulling model: {model}")
    exit_code = os.system(f"ollama pull {model}")
    if exit_code != 0:
        print(f"Warning: failed to pull {model} (exit code {exit_code})")

def ollama_list_models() -> List[Dict[str, Any]]:
    """List installed Ollama models via HTTP."""
    resp = requests.get(f"{OLLAMA_HOST}/api/tags")
    resp.raise_for_status()
    return resp.json().get("models", [])

def ollama_embed(texts: List[str], model: str = EMBEDDING_MODEL) -> List[List[float]]:
    """Get embeddings from Ollama /api/embed."""
    payload = {
        "model": model,
        "input": texts,
    }
    resp = requests.post(f"{OLLAMA_HOST}/api/embed", json=payload)
    resp.raise_for_status()
    data = resp.json()
    return data["embeddings"]

def ollama_chat(prompt: str, model: str = CHAT_MODEL, system_prompt: str | None = None) -> str:
    """Single-turn chat via Ollama /api/chat."""
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})

    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
    }
    resp = requests.post(f"{OLLAMA_HOST}/api/chat", json=payload)
    resp.raise_for_status()
    data = resp.json()
    return data["message"]["content"]

def ollama_delete_model(model: str):
    """Delete an Ollama model via HTTP."""
    print(f"Deleting Ollama model: {model}")
    resp = requests.delete(f"{OLLAMA_HOST}/api/delete", json={"name": model})
    if resp.status_code != 200:
        print(f"Failed to delete {model}: {resp.status_code} {resp.text}")
    else:
        print(f"Deleted {model}")


## Pull models in Ollama

In [7]:
ollama_pull_model(EMBEDDING_MODEL)
ollama_pull_model(CHAT_MODEL)

Pulling model: nomic-embed-text


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling 970aa74c0a90: 100% ▕██████████████████▏ 274 MB                         
pulling c71d239df917: 100% ▕██████████████████▏  11 KB                         
pulling ce4a164fc046: 100% ▕██████████████████▏   17 B                         
pulling 31df23ea7daa: 100% ▕██████████████████▏  420 B                         
verifying sha256 digest 
writing manifest 
success 
pulling manifest ⠋ 

Pulling model: llama3:8b


pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling 6a0746a1ec1a: 100% ▕██████████████████▏ 4.7 GB                         
pulling 4fa551d4f938: 100% ▕██████████████████▏  12 KB                         
pulling 8ab4849b038c: 100% ▕██████████████████▏  254 B                         
pulling 577073ffcc6c: 100% ▕██████████████████▏  110 B                         
pulling 3f8eb4da87fa: 100% ▕██████████████████▏  485 B                         
verifying sha256 digest 
writing manifest 
success 


## List installed Ollama models

In [8]:
print("\nInstalled models:")
for m in ollama_list_models():
    print(" -", m["name"])


Installed models:
 - llama3:8b
 - nomic-embed-text:latest


## Create/reset the Weaviate collection (BYO vectors) for RAG
- We are not using the built-in vectorizer (in Weaviate)
- We get embeddings from Ollama, i.e. Bring Your Own (BYO) vectors.

In [ ]:
# Delete collection if it exists, so the notebook is repeatable
if client.collections.exists(COLLECTION_NAME):
    print(f"Collection {COLLECTION_NAME} already exists - deleting it.")
    client.collections.delete(COLLECTION_NAME)

rag_collection = client.collections.create(
    name=COLLECTION_NAME,
    properties=[
        Property(name="title", data_type=DataType.TEXT),
        Property(name="content", data_type=DataType.TEXT),
    ],
    # We provide our own vectors from Ollama (BYO vectors)
    vector_config=Configure.Vectors.self_provided(),
)

print("Created collection:", COLLECTION_NAME)

Created collection: Group1RagDocument


## Sample documents for RAG

In [15]:
documents = [
    {
        "title": "Ollama Overview",
        "content": """
        Ollama is a tool for running large language models locally.
        It provides a simple CLI and HTTP API to pull, run, and manage models.
        You can use it for chat, embeddings, and more.
        """
    },
    {
        "title": "Weaviate Overview",
        "content": """
        Weaviate is an open source vector database.
        It stores objects and vectors, and provides similarity search.
        You can use it for semantic search and retrieval augmented generation (RAG).
        """
    },
    {
        "title": "RAG Pattern",
        "content": """
        Retrieval Augmented Generation (RAG) is a pattern where you:
        1) embed documents and store them in a vector database,
        2) embed a user query,
        3) retrieve the most similar documents,
        4) feed them into a language model as context for generation.
        """
    },
]

for d in documents:
    d["content"] = textwrap.dedent(d["content"]).strip()

documents

[{'title': 'Ollama Overview',
  'content': 'Ollama is a tool for running large language models locally.\nIt provides a simple CLI and HTTP API to pull, run, and manage models.\nYou can use it for chat, embeddings, and more.'},
 {'title': 'Weaviate Overview',
  'content': 'Weaviate is an open source vector database.\nIt stores objects and vectors, and provides similarity search.\nYou can use it for semantic search and retrieval augmented generation (RAG).'},
 {'title': 'RAG Pattern',
  'content': 'Retrieval Augmented Generation (RAG) is a pattern where you:\n1) embed documents and store them in a vector database,\n2) embed a user query,\n3) retrieve the most similar documents,\n4) feed them into a language model as context for generation.'}]

## Embed documents with Ollama and insert into Weaviate Database

In [16]:
texts = [doc["content"] for doc in documents]
embeddings = ollama_embed(texts, model=EMBEDDING_MODEL)

print(f"Created {len(embeddings)} embeddings, dim={len(embeddings[0])}")

rag_collection = client.collections.get(COLLECTION_NAME)

for doc, vec in zip(documents, embeddings):
    rag_collection.data.insert(
        properties={
            "title": doc["title"],
            "content": doc["content"],
        },
        vector=vec,
    )

print("Inserted documents into Weaviate collection:", COLLECTION_NAME)

Created 3 embeddings, dim=768
Inserted documents into Weaviate collection: Group1RagDocument


## List stored objects in Weaviate (verify)
- You should see three documents.

In [17]:
rag_collection = client.collections.get(COLLECTION_NAME)

response = rag_collection.query.fetch_objects(
    limit=10
)

print("Stored objects:\n")
for obj in response.objects:
    print("ID:", obj.uuid)
    print("Title:", obj.properties.get("title"))
    print("Content snippet:", obj.properties.get("content", "")[:120], "...")
    print("-" * 60)

Stored objects:

ID: 05e88314-272d-4328-af57-94c45c43449e
Title: RAG Pattern
Content snippet: Retrieval Augmented Generation (RAG) is a pattern where you:
1) embed documents and store them in a vector database,
2)  ...
------------------------------------------------------------
ID: a9b1d087-d8f4-445a-ad61-ee3406fa90e7
Title: Ollama Overview
Content snippet: Ollama is a tool for running large language models locally.
It provides a simple CLI and HTTP API to pull, run, and mana ...
------------------------------------------------------------
ID: aaf2ac6a-5574-4465-9111-bbe0ab66f5dd
Title: Weaviate Overview
Content snippet: Weaviate is an open source vector database.
It stores objects and vectors, and provides similarity search.
You can use i ...
------------------------------------------------------------


## Query Weaviate using a question embedding (near_vector), i.e. semantic search
- You should see the most relevant documents (likely “RAG Pattern” and “Weaviate Overview”).

In [18]:
question = "How can I build a RAG system using a local LLM and a vector database?"
print("User question:", question)

q_vec = ollama_embed([question], model=EMBEDDING_MODEL)[0]

rag_collection = client.collections.get(COLLECTION_NAME)

search_result = rag_collection.query.near_vector(
    near_vector=q_vec,
    limit=3,
)

print("Search hits:\n")
for i, obj in enumerate(search_result.objects, start=1):
    print(f"[Hit {i}]")
    print("Title:", obj.properties.get("title"))
    print("Content snippet:", obj.properties.get("content", "")[:200], "...")
    print()

User question: How can I build a RAG system using a local LLM and a vector database?
Search hits:

[Hit 1]
Title: RAG Pattern
Content snippet: Retrieval Augmented Generation (RAG) is a pattern where you:
1) embed documents and store them in a vector database,
2) embed a user query,
3) retrieve the most similar documents,
4) feed them into a  ...

[Hit 2]
Title: Weaviate Overview
Content snippet: Weaviate is an open source vector database.
It stores objects and vectors, and provides similarity search.
You can use it for semantic search and retrieval augmented generation (RAG). ...

[Hit 3]
Title: Ollama Overview
Content snippet: Ollama is a tool for running large language models locally.
It provides a simple CLI and HTTP API to pull, run, and manage models.
You can use it for chat, embeddings, and more. ...



## Build RAG prompt (retrieved docs as context) and call chat model
- Now we use the retrieved documents as context for the chat model.
- You should get a coherent explanation of how to build a RAG system, grounded in the retrieved documents.

In [19]:
hits = search_result.objects

context_parts = []
for i, obj in enumerate(hits, start=1):
    title = obj.properties.get("title", "Untitled")
    content = obj.properties.get("content", "")
    context_parts.append(f"[Document {i} - {title}]\n{content}")

context_text = "\n\n".join(context_parts)

system_prompt = """
You are a helpful assistant. Use ONLY the provided context to answer the user's question.
If the answer is not in the context, say that you don't know.
"""

rag_prompt = f"""
Context:
{context_text}

User question:
{question}

Answer based only on the context above.
"""

answer = ollama_chat(
    prompt=rag_prompt.strip(),
    model=CHAT_MODEL,
    system_prompt=system_prompt.strip(),
)

print("RAG answer:\n")
print(answer)

RAG answer:

Based on the provided context, here's how you can build a RAG system using a local LLM and a vector database:

1. Embed documents and store them in the Weaviate vector database (step 1 from Document 1 - RAG Pattern).
2. Embed a user query.
3. Retrieve the most similar documents from the Weaviate vector database (step 3 from Document 1 - RAG Pattern).
4. Feed the retrieved documents into an Ollama-local LLM as context for generation (step 4 from Document 1 - RAG Pattern).

You can use Ollama to run a large language model locally, and then use Weaviate as your vector database to store and retrieve the embedded documents.


## Remove all data (embeddings) from Weaviate (drop collection)
- You should not see any documents in the Weaviate database.

In [20]:
if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)
    print(f"Deleted collection {COLLECTION_NAME} (and all its objects).")

print("Collection exists after delete?", client.collections.exists(COLLECTION_NAME))

Deleted collection Group1RagDocument (and all its objects).
Collection exists after delete? False


## Delete Ollama models, then list existing models
- This will delete the two models (embedding and chat) from the Ollama instance.
- Don't run the cell below if you want to keep them.

In [21]:
ollama_delete_model(EMBEDDING_MODEL)
ollama_delete_model(CHAT_MODEL)

print("\nOllama models after deletion:")
for m in ollama_list_models():
    print(" -", m["name"])

Deleting Ollama model: nomic-embed-text
Deleted nomic-embed-text
Deleting Ollama model: llama3:8b
Deleted llama3:8b

Ollama models after deletion:


## Fetch data for you RAG documents from your webapp database
- Enter the `database`, `username`, and `password` for your group's webapp database in the cell below, before executing the cell.
- Also change the SELECT statament `SELECT TOP 5 * FROM WeatherForecasts` if you have removed the table `WeatherForecasts`.

In [ ]:
server = "172.31.42.71,1433"
database = "group1"
username = "group1"
password = "<enter password here>"

# Note the 'mssql+pyodbc' and 'driver=ODBC Driver 18 for SQL Server'
connection_url = (
    f"mssql+pyodbc://{username}:{password}"
    f"@{server}/{database}"
    "?driver=ODBC+Driver+18+for+SQL+Server"
    "&Encrypt=yes"
    "&TrustServerCertificate=yes"
)

engine = create_engine(connection_url)

with engine.connect() as conn:
    result = conn.execute(text("SELECT TOP 5 * FROM WeatherForecasts"))
    for row in result:
        print(row)

(1, datetime.date(2026, 6, 30), 48, 'Seeded')
(2, datetime.date(2026, 7, 1), 18, 'Seeded')
(3, datetime.date(2026, 7, 2), 2, 'Seeded')
(4, datetime.date(2026, 7, 3), 51, 'Seeded')
(5, datetime.date(2026, 7, 4), 1, 'Seeded')
